# GigScore — 01: Data Exploration

**Sprint 2** | Schema version: `2.0.0`

Objective: Profile raw datasets, validate against `feature_schema.json`, and surface data quality issues before building the ingestion pipeline.

## 1. Environment Setup

In [ ]:
import sys
import json
from pathlib import Path

# Ensure E:\gigworker is on sys.path so package imports resolve
ROOT = Path("E:/gigworker")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from ml.utils.paths import get_data_path, ensure_directories
from ml.utils.feature_engineering import build_feature_vector, validate_features

ensure_directories()
print('Environment ready.')

## 2. Load Raw Datasets

In [ ]:
txn_path     = get_data_path('worker_transactions.csv', processed=False)
profile_path = get_data_path('worker_profiles.csv',     processed=False)
loans_path   = get_data_path('loans.csv',               processed=False)

df_txn     = pd.read_csv(txn_path)     if txn_path.exists()     else pd.DataFrame()
df_profile = pd.read_csv(profile_path) if profile_path.exists() else pd.DataFrame()
df_loans   = pd.read_csv(loans_path)   if loans_path.exists()   else pd.DataFrame()

print(f'Transactions : {df_txn.shape}')
print(f'Profiles     : {df_profile.shape}')
print(f'Loans        : {df_loans.shape}')

## 3. Schema Inspection

In [ ]:
schema_path = ROOT / 'ml' / 'models' / 'feature_schema.json'
with schema_path.open() as f:
    schema = json.load(f)

expected_features = [
    name
    for group in schema['feature_groups'].values()
    for name in group['features']
]

print(f'Schema version   : {schema["version"]}')
print(f'Total features   : {len(expected_features)}')
print(f'Feature groups   : {list(schema["feature_groups"].keys())}')

## 4. Missing Value Analysis

In [ ]:
for name, df in [('Transactions', df_txn), ('Profiles', df_profile), ('Loans', df_loans)]:
    if df.empty:
        print(f'{name}: no data loaded')
        continue
    null_pct = df.isnull().mean().sort_values(ascending=False)
    high_null = null_pct[null_pct > 0.05]
    print(f'\n{name} — columns with >5% nulls:')
    print(high_null.to_string() if not high_null.empty else '  None')

## 5. Duplicate Detection

In [ ]:
for name, df, key in [
    ('Transactions', df_txn,     'transaction_id'),
    ('Profiles',     df_profile, 'worker_id'),
    ('Loans',        df_loans,   'worker_id'),
]:
    if df.empty:
        continue
    dup_rows = df.duplicated().sum()
    dup_keys = df[key].duplicated().sum() if key in df.columns else 'key missing'
    print(f'{name}: {dup_rows} duplicate rows | {dup_keys} duplicate {key}s')

## 6. Outlier Summary

In [ ]:
def iqr_outlier_count(series: pd.Series) -> int:
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    return int(((series < q1 - 1.5 * iqr) | (series > q3 + 1.5 * iqr)).sum())

for name, df in [('Transactions', df_txn), ('Profiles', df_profile)]:
    if df.empty:
        continue
    num_cols = df.select_dtypes(include='number').columns
    outlier_counts = {col: iqr_outlier_count(df[col].dropna()) for col in num_cols}
    print(f'\n{name} outlier counts (IQR method):')
    for col, cnt in sorted(outlier_counts.items(), key=lambda x: -x[1]):
        if cnt > 0:
            print(f'  {col}: {cnt}')

## 7. Correlation Placeholder

In [ ]:
# Populate once processed feature vectors are available in Sprint 3.
# Replace df_features with the output of build_feature_vector applied to all workers.
#
# df_features = pd.DataFrame([...])  # Sprint 3
# corr = df_features.corr()
# plt.figure(figsize=(14, 10))
# plt.imshow(corr, cmap='coolwarm', aspect='auto')
# plt.colorbar()
# plt.xticks(range(len(corr)), corr.columns, rotation=90)
# plt.yticks(range(len(corr)), corr.columns)
# plt.title('Feature Correlation Matrix')
# plt.tight_layout()
# plt.show()
print('Correlation matrix — Sprint 3 (requires processed feature vectors)')

## 8. Feature Engineering Roadmap

| Feature | Source Column(s) | Transform | Sprint |
|---|---|---|---|
| `monthly_income_avg` | `amount` (credit rows) | 3-month rolling mean | 3 |
| `income_volatility` | `amount` (credit rows) | `std / mean` | 3 |
| `active_day_ratio` | `transaction_date` | `active_days / 90` | 3 |
| `emi_ratio` | `amount` (debit EMI rows) | `emi_sum / income_avg` | 3 |
| `cash_flow_stability` | `amount` | `months_positive / total_months` | 3 |
| `fraud_indicators` | `anomaly_flag` | sum count | 3 |
| All service quality features | Platform API | Future integration | 5 |

**Next step:** implement `data/loaders.py` (Sprint 3) to populate `data/processed/`.

## 9. build_feature_vector Smoke Test

Verifies that the feature engineering module resolves correctly.

In [ ]:
sample_raw = {
    'monthly_income_avg': 32000,
    'income_volatility': 0.22,
    'active_day_ratio': 0.71,
    'platform_tenure_months': 18,
    'average_rating': 4.3,
    'emi_ratio': 0.18,
    'bill_payment_ratio': 0.92,
    'fraud_indicators': 0,
    'identity_verified': True,
    'document_verified': True,
}

vector = build_feature_vector(sample_raw)
is_valid = validate_features(vector)

print(f'Feature vector keys : {len(vector)}')
print(f'Validation passed   : {is_valid}')